In [1]:
import numpy as np
import pandas as pd
import os
import concurrent.futures

# Create a clean directory for our files
os.makedirs("Simulation_Data", exist_ok=True)

# 1. Define the parameters from the research paper table
datasets_info = {
    "Dataset_I":   {"n_sample": 60,  "n_feature": 100,   "beta_type": "manual"},
    "Dataset_II":  {"n_sample": 120, "n_feature": 1000,  "beta_type": "random"},
    "Dataset_III": {"n_sample": 240, "n_feature": 10000, "beta_type": "random"},
    "Dataset_IV":  {"n_sample": 480, "n_feature": 10000, "beta_type": "random"}
}

sigma = 3

# Define 8 specific seeds to generate 8 reproducible versions of each dataset
seeds = [0,1,2,3,4,5,6,7,8,9]

def generate_dataset(args):
    name, info, seed_idx, seed_val = args
    
    n = info["n_sample"]
    p = info["n_feature"]
    
    # Initialize random generator INSIDE the function for thread safety
    rs = np.random.default_rng(seed_val)
    
    # 2. Generate Beta (True Coefficients)
    if info["beta_type"] == "manual":
        specific_betas = [3, 3, -3, 2, 2, -2, 1.5, 1.5, 1.5, -1.5]
        beta = np.hstack([specific_betas, np.zeros(p - 10)])
    else:
        beta = np.hstack([rs.normal(loc=0, scale=2, size=50), np.zeros(p - 50)])
        
    # 3 & 4. Optimized Generation (Block-by-Block)
    X = np.zeros((n, p))
    
    # Create just the 5x5 covariance block once
    block_corr = np.full((5, 5), 0.9)
    np.fill_diagonal(block_corr, 1)
    block_cov = (sigma ** 2) * block_corr
    
    # Generate data iteratively for each independent block
    for i in range(int(p / 5)):
        X[:, 5 * i: 5 * (i + 1)] = rs.multivariate_normal(
            mean=np.zeros(5), 
            cov=block_cov, 
            size=n
        )
        
    epsilon = rs.normal(loc=0, scale=sigma, size=n)
    
    # 5. Generate y
    y = X @ beta + epsilon
    
    # 6. Save directly to CSVs (appending the version number)
    version = seed_val
    pd.DataFrame(X).to_csv(f'Simulation_Data/X_{name}_v{version}.csv', index=False)
    pd.DataFrame(y).to_csv(f'Simulation_Data/y_{name}_v{version}.csv', index=False)
    
    # New: Save the exact true Betas for the RME calculations
    pd.DataFrame(beta).to_csv(f'Simulation_Data/beta_{name}_v{version}.csv', index=False)
    
    return f"Success: {name} (Version {version}) saved! X shape: {X.shape}, y shape: {y.shape}"

# --- Multiprocessing Execution ---
if __name__ == '__main__':
    # Create a task list of all 16 combinations (4 datasets * 4 seeds)
    tasks = []
    for name, info in datasets_info.items():
        for seed_idx, seed_val in enumerate(seeds):
            tasks.append((name, info, seed_idx, seed_val))

    # ProcessPoolExecutor automatically detects and uses all available CPU cores
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = executor.map(generate_dataset, tasks)
        
        for res in results:
            print(res)
            
    print("\nAll dataset versions and true betas generated and saved successfully.")

Success: Dataset_I (Version 0) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 1) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 2) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 3) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 4) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 5) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 6) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 7) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 8) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_I (Version 9) saved! X shape: (60, 100), y shape: (60,)
Success: Dataset_II (Version 0) saved! X shape: (120, 1000), y shape: (120,)
Success: Dataset_II (Version 1) saved! X shape: (120, 1000), y shape: (120,)
Success: Dataset_II (Version 2) saved! X shape: (120, 1000), y shape: (120,)
Success: Dataset_II (Version 3) saved! 